# TRIBE v2 Demo: Predicting Brain Responses to Naturalistic Stimuli

[TRIBE v2](https://github.com/facebookresearch/tribev2) is a deep multimodal brain encoding model that predicts **fMRI brain responses** to naturalistic stimuli — video, audio, and text.

It combines state-of-the-art feature extractors — **LLaMA 3.2** (text), **V-JEPA2** (video), and **Wav2Vec-BERT** (audio) — into a unified Transformer that maps multimodal representations onto the cortical surface (**fsaverage5**, ~20k vertices).

In this notebook, we will:
1. Load a pretrained TRIBE v2 model from HuggingFace
2. Predict brain responses to a **video** clip
3. Predict brain responses to **audio** generated from text
4. Visualize the predicted activity on a 3D brain surface

## Setup (for Colab users)

1. Activate the GPU (Menu > Runtime > Change runtime)
2. Run the command below
3. Restart your environment for the new packages to be taken into account

In [1]:
!nvidia-smi

Thu Sep 10 18:21:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!uv pip install "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

Using Python 3.13.15 environment at: /usr
Resolved 137 packages in 8.62s
Prepared 45 packages in 53.60s
Uninstalled 22 packages in 1.20s
Installed 45 packages in 449ms
 + cyclopts==4.25.2
 - decorator==4.4.2
 + decorator==5.3.1
 + exca==0.5.20
 + gtts==2.2.4
 + julius==0.2.8
 + langdetect==1.0.9
 + levenshtein==0.27.4
 + lightning-utilities==0.15.3
 + mne==1.13.0
 + mne-bids==0.19.0
 - moviepy==1.0.3
 + moviepy==2.2.1
 + neuralset==0.0.2
 + neuraltrain==0.0.2
 + nilearn==0.14.1
 - numpy==2.1.3
 + numpy==2.2.6
 - nvidia-cublas-cu12==12.8.4.1
 + nvidia-cublas-cu12==12.4.5.8
 - nvidia-cuda-cupti-cu12==12.8.90
 + nvidia-cuda-cupti-cu12==12.4.127
 - nvidia-cuda-nvrtc-cu12==12.8.93
 + nvidia-cuda-nvrtc-cu12==12.4.127
 - nvidia-cuda-runtime-cu12==12.8.90
 + nvidia-cuda-runtime-cu12==12.4.127
 - nvidia-cudnn-cu12==9.19.0.56
 + nvidia-cudnn-cu12==9.1.0.70
 - nvidia-cufft-cu12==11.3.3.83
 + nvidia-cufft-cu12==11.2.1.3
 - nvidia-curand-cu12==10.3.9.90
 + nvidia-curand-cu12==10.3.5.147
 - nvidia-c

In [3]:
!pip uninstall -y torchaudio
!pip install --no-cache-dir torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.3 MB/s eta 0:00:00


In [1]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torchvision
import torchaudio

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        "GB"
    )

!nvidia-smi

Torch: 2.6.0+cu124
Torchvision: 0.21.0+cu124
Torchaudio: 2.6.0+cu124
CUDA: True
GPU: Tesla T4
VRAM: 14.56317138671875 GB
Thu Sep 10 18:30:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                  

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [7]:
!ls "/content/drive/MyDrive/TRIBE"

Movie1.avi


In [8]:
import shutil
from pathlib import Path

drive_video = Path("/content/drive/MyDrive/TRIBE/Movie1.avi")
video_path = Path("/content/cache/Movie1.avi")

video_path.parent.mkdir(parents=True, exist_ok=True)

shutil.copy2(
    drive_video,
    video_path
)

print("Kopyalama tamamlandı.")
print("Local path:", video_path)
print("Dosya var mı:", video_path.exists())

Kopyalama tamamlandı.
Local path: /content/cache/Movie1.avi
Dosya var mı: True


In [9]:
from moviepy import VideoFileClip

clip = VideoFileClip(str(video_path))

print(f"Süre: {clip.duration:.2f} saniye")
print(f"Dakika: {clip.duration / 60:.2f}")

clip.close()

Süre: 339.44 saniye
Dakika: 5.66


Loading the model

In [10]:
from tribev2.demo_utils import TribeModel
from pathlib import Path

CACHE_FOLDER = Path("./cache")

model = TribeModel.from_pretrained(
    "facebook/tribev2",
    cache_folder=CACHE_FOLDER,
)

model.data.num_workers = 0

print("Model hazır.")

/usr/local/lib/python3.13/dist-packages/neuralset/extractors/base.py:707: UserWarning: LabelEncoder: event_types has not been set, are you sure you want to apply this extractor to all events?
  warnings.warn(
2026-09-10 19:27:03 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.


config.yaml:   0%|          | 0.00/18.0k [00:00<?, ?B/s]

best.ckpt: reconstructing file:   0%|          |  0.00B /  709MB            

best.ckpt: downloading bytes:           |  0.00B            

2026-09-10 19:27:08 - WARNING - neuralset.extractors.base:798 - Missing events will be encoded using the default all-zero value (for example, 0 or a zero vector/tensor), which may be indistinguishable from a valid class if that class is also mapped to zeros. Set treat_missing_as_separate_class=True to avoid this.
INFO - Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
INFO:tribev2.demo_utils:Loading model from /root/.cache/huggingface/hub/models--facebook--tribev2/snapshots/f894e783020944dcd96e5568550afe2aa9743f9f/best.ckpt
/usr/local/lib/python3.13/dist-packages/x_transformers/x_transformers.py:439: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)
/usr/local/lib/python3.13/dist-packages/x_transformers/x_transformers.py:461: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use 

Model hazır.


Event Dataframe

In [11]:
import time

event_start = time.time()

df = model.get_events_dataframe(
    video_path=video_path
)

event_time = time.time() - event_start

display(df)

print("Event sayısı:", len(df))
print(f"Event preprocessing: {event_time:.2f} saniye")

Extract audio from video events: 100%|██████████| 1/1 [00:00<00:00, 22.89it/s]
Extracting words from audio: 0it [00:00, ?it/s]
2026-09-10 19:29:14 - INFO - neuralset.events.transforms.text:56 - No Word events found, skipping
INFO:neuralset.events.transforms.text:No Word events found, skipping
2026-09-10 19:29:14 - INFO - neuralset.events.transforms.text:175 - No Word events found, skipping
INFO:neuralset.events.transforms.text:No Word events found, skipping
Add context to words: 0it [00:00, ?it/s]


,type,start,duration,timeline,subject,session,task,run,filepath,frequency,offset,stop,context
0,Video,0.0,60.00,default,default,,,,/content/cache/Movie1.avi,29.97003,0.0,60.00,NaN
1,Video,60.0,60.00,default,default,,,,/content/cache/Movie1.avi,29.97003,60.0,120.00,NaN
2,Video,120.0,60.00,default,default,,,,/content/cache/Movie1.avi,29.97003,120.0,180.00,NaN
3,Video,180.0,60.00,default,default,,,,/content/cache/Movie1.avi,29.97003,180.0,240.00,NaN
4,Video,240.0,60.00,default,default,,,,/content/cache/Movie1.avi,29.97003,240.0,300.00,NaN
5,Video,300.0,39.44,default,default,,,,/content/cache/Movie1.avi,29.97003,300.0,339.44,NaN


Event sayısı: 6
Event preprocessing: 0.40 saniye


### Run the model

We feed the events dataframe to `model.predict()`, which extracts features for each modality, runs them through the Transformer, and returns predicted brain activity.

NOTE: you will have to request access to the Llama-3.2 model using your HuggingFace account.

The output `preds` has shape `(n_timesteps, n_vertices)` — one prediction per second of stimulus, with ~20k cortical vertices. The `segments` list contains the corresponding time segments with their associated events.

In [ ]:
import torch
import time

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

start = time.time()

with torch.inference_mode(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    preds, segments = model.predict(events=df)

elapsed = time.time() - start
peak_vram = torch.cuda.max_memory_allocated() / 1024**3

print(f"Inference süresi: {elapsed/60:.2f} dakika")
print(f"Peak VRAM: {peak_vram:.2f} GB")
print(f"Predictions shape: {preds.shape}")
print(f"Segment sayısı: {len(segments)}")

[19:31:52 WARNING] Removing extractor audio as there are no corresponding events
[19:31:52 WARNING] Removing extractor text as there are no corresponding events
[19:31:52 INFO] Preparing extractor: video
INFO:tribev2.main:Preparing extractor: video
  0%|          | 0/6 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.14GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/843 [00:00<?, ?it/s]

video_preprocessor_config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

2026-09-10 19:32:39 - DEBUG - neuralset.extractors.video:277 - Loaded Video (duration 60.0s at 29.97002997002997fps, shape (720, 480)):
/content/cache/Movie1.avi
DEBUG:neuralset.extractors.video:Loaded Video (duration 60.0s at 29.97002997002997fps, shape (720, 480)):
/content/cache/Movie1.avi

Encoding video:   0%|          | 0/120 [00:00<?, ?it/s]2026-09-10 19:32:46 - DEBUG - neuralset.extractors.video:311 - Created Tensor with size (120, 20, 1408)
DEBUG:neuralset.extractors.video:Created Tensor with size (120, 20, 1408)

Encoding video:  40%|████      | 48/120 [03:53<05:46,  4.82s/it]